<a href="https://colab.research.google.com/github/aisha13dikko-sudo/using-synthetic-data-for-thermal-comfort-classification/blob/main/wk6_paramodel_timeseries_autotherm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install deepecho --quiet 2>&1 | tail -15
print("Checking installation...")
import deepecho
print(f"✅ DeepEcho installed: {deepecho.__version__}")

Checking installation...
✅ DeepEcho installed: 0.8.1


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import re

from datasets import load_dataset
from deepecho import PARModel

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, balanced_accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

print("✅ Imports ready")

✅ Imports ready


In [3]:
!pip install datasets --quiet

In [4]:
print("Loading AutoTherm indoor dataset...")
dataset = load_dataset('kopetri/AutoTherm', 'indoor')
train_df = dataset['train'].to_pandas()

def extract_participant_id(filename):
    match = re.search(r'participant_\d+', filename)
    return match.group() if match else 'unknown'

train_df['participant_id'] = train_df['file_name'].apply(extract_participant_id)

test_participants  = ['participant_16', 'participant_14', 'participant_20']
train_participants = [p for p in train_df['participant_id'].unique()
                      if p not in test_participants]

print(f"Train participants: {sorted(train_participants)}")
print(f"Test participants:  {sorted(test_participants)}")
print(f"Total rows: {len(train_df):,}")

Loading AutoTherm indoor dataset...


README.md:   0%|          | 0.00/8.57k [00:00<?, ?B/s]

indoor/train-00000-of-00002.parquet:   0%|          | 0.00/29.8M [00:00<?, ?B/s]

indoor/train-00001-of-00002.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

indoor/test-00000-of-00001.parquet:   0%|          | 0.00/7.41M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1566728 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/194829 [00:00<?, ? examples/s]

Train participants: ['participant_10', 'participant_11', 'participant_15', 'participant_17', 'participant_18', 'participant_19', 'participant_2', 'participant_21', 'participant_3', 'participant_4', 'participant_6', 'participant_7', 'participant_8']
Test participants:  ['participant_14', 'participant_16', 'participant_20']
Total rows: 1,566,728


In [5]:
# Check that Timestamp actually gives us a usable time order
print("Timestamp dtype:", train_df['Timestamp'].dtype)
print("\nSample timestamps for one participant:")
sample_participant = train_df[train_df['participant_id'] == 'participant_2']
print(sample_participant[['file_name', 'Timestamp']].head(10))

# Check if rows are already in time order, or need sorting
print("\nAre rows already sorted by Timestamp within this participant?")
sorted_check = sample_participant['Timestamp'].is_monotonic_increasing
print(f"Monotonic increasing: {sorted_check}")

Timestamp dtype: object

Sample timestamps for one participant:
                                                file_name  \
992736  frontal_participant_2_2022-04-22_16-28-16/fram...   
992737  frontal_participant_2_2022-04-22_16-28-16/fram...   
992738  frontal_participant_2_2022-04-22_16-28-16/fram...   
992739  frontal_participant_2_2022-04-22_16-28-16/fram...   
992740  frontal_participant_2_2022-04-22_16-28-16/fram...   
992741  frontal_participant_2_2022-04-22_16-28-16/fram...   
992742  frontal_participant_2_2022-04-22_16-28-16/fram...   
992743  frontal_participant_2_2022-04-22_16-28-16/fram...   
992744  frontal_participant_2_2022-04-22_16-28-16/fram...   
992745  frontal_participant_2_2022-04-22_16-28-16/fram...   

                         Timestamp  
992736  2022-04-22 16:31:25.660165  
992737  2022-04-22 16:31:25.684098  
992738  2022-04-22 16:31:25.740921  
992739  2022-04-22 16:31:25.780843  
992740  2022-04-22 16:31:25.794802  
992741  2022-04-22 16:31:25.878554  
99274

In [6]:
# Convert Timestamp from text to proper datetime
train_df['Timestamp'] = pd.to_datetime(train_df['Timestamp'])

# Sort each participant's data by time, just to be certain
train_df = train_df.sort_values(['participant_id', 'Timestamp']).reset_index(drop=True)

# Define the columns we'll keep for time-series modelling
# Same feature set as before, plus we need participant_id as the
# "entity" column DeepEcho uses to know where one sequence ends
# and another begins
ts_feature_cols = [
    'Age', 'Gender', 'Weight', 'Height', 'Bodyfat', 'Bodytemp',
    'Sport-Last-Hour', 'Time-Since-Meal', 'Tiredness', 'Clothing-Level',
    'Radiation-Temp', 'PCE-Ambient-Temp', 'Wrist_Skin_Temperature',
    'Heart_Rate', 'GSR', 'Ambient_Temperature', 'Ambient_Humidity',
    'Solar_Radiation', 'Label'
]

# Encode Gender for consistency
train_df['Gender'] = LabelEncoder().fit_transform(train_df['Gender'].astype(str))

# Build fixed 10-step windows per participant
def build_windows(df, participant_list, window_size=10):
    windows = []
    seq_id = 0
    for pid in participant_list:
        p_data = df[df['participant_id'] == pid][ts_feature_cols].reset_index(drop=True)
        n_windows = len(p_data) // window_size
        for w in range(n_windows):
            window = p_data.iloc[w*window_size : (w+1)*window_size].copy()
            window['sequence_id'] = f"{pid}_seq{w}"
            window['participant_id'] = pid
            window['time_step'] = range(window_size)
            windows.append(window)
        seq_id += 1
    return pd.concat(windows, axis=0).reset_index(drop=True)

print("Building 10-step sequences for training participants...")
train_sequences = build_windows(train_df, train_participants, window_size=10)
print(f"✅ Built {train_sequences['sequence_id'].nunique():,} training sequences")
print(f"Total rows: {len(train_sequences):,}")

print("\nBuilding 10-step sequences for test participants...")
test_sequences = build_windows(train_df, test_participants, window_size=10)
print(f"✅ Built {test_sequences['sequence_id'].nunique():,} test sequences")
print(f"Total rows: {len(test_sequences):,}")

print("\nExample sequence:")
print(train_sequences[train_sequences['sequence_id'] == train_sequences['sequence_id'].iloc[0]][
    ['sequence_id', 'time_step', 'Ambient_Temperature', 'Wrist_Skin_Temperature', 'Label']
])

Building 10-step sequences for training participants...
✅ Built 127,666 training sequences
Total rows: 1,276,660

Building 10-step sequences for test participants...
✅ Built 29,000 test sequences
Total rows: 290,000

Example sequence:
           sequence_id  time_step  Ambient_Temperature  \
0  participant_10_seq0          0                 20.4   
1  participant_10_seq0          1                 20.4   
2  participant_10_seq0          2                 20.4   
3  participant_10_seq0          3                 20.4   
4  participant_10_seq0          4                 20.4   
5  participant_10_seq0          5                 20.4   
6  participant_10_seq0          6                 20.4   
7  participant_10_seq0          7                 20.4   
8  participant_10_seq0          8                 20.4   
9  participant_10_seq0          9                 20.4   

   Wrist_Skin_Temperature  Label  
0                   30.05     -2  
1                   30.05     -2  
2                   3

In [7]:
# How much does the data actually CHANGE within a typical 10-step window?
# If it's near-zero, time-series modelling won't have meaningful signal to exploit

variation_check = train_sequences.groupby('sequence_id').agg({
    'Ambient_Temperature': lambda x: x.max() - x.min(),
    'Wrist_Skin_Temperature': lambda x: x.max() - x.min(),
    'GSR': lambda x: x.max() - x.min(),
    'Label': lambda x: x.nunique()
}).rename(columns={
    'Ambient_Temperature': 'AmbTemp_range',
    'Wrist_Skin_Temperature': 'SkinTemp_range',
    'GSR': 'GSR_range',
    'Label': 'unique_labels_in_window'
})

print("How much do features vary WITHIN a 10-step window, on average?")
print(variation_check.describe())

print(f"\nWindows where Label changes at all within the window: "
      f"{(variation_check['unique_labels_in_window'] > 1).sum():,} "
      f"out of {len(variation_check):,} "
      f"({(variation_check['unique_labels_in_window'] > 1).mean()*100:.2f}%)")

How much do features vary WITHIN a 10-step window, on average?
       AmbTemp_range  SkinTemp_range      GSR_range  unique_labels_in_window
count  127666.000000   127666.000000  127666.000000            127666.000000
mean        0.005623        0.002814       0.005462                 1.001606
std         0.024227        0.010977       0.040367                 0.040040
min         0.000000        0.000000       0.000000                 1.000000
25%         0.000000        0.000000       0.000000                 1.000000
50%         0.000000        0.000000       0.000000                 1.000000
75%         0.000000        0.000000       0.000000                 1.000000
max         0.300000        0.680000       2.578362                 2.000000

Windows where Label changes at all within the window: 205 out of 127,666 (0.16%)


In [8]:
# What is the actual sampling interval?
sample_p = train_df[train_df['participant_id'] == 'participant_10'].reset_index(drop=True)
time_diffs = sample_p['Timestamp'].diff().dt.total_seconds()

print("Time between consecutive readings (seconds):")
print(time_diffs.describe())
print(f"\nMedian gap: {time_diffs.median():.4f} seconds")
print(f"That means roughly {1/time_diffs.median():.1f} readings per second")

# How many rows would we need for a genuine 10-second window?
rows_per_10_sec = int(10 / time_diffs.median())
print(f"\nTo span 10 real seconds, we need approximately {rows_per_10_sec} rows per window")

Time between consecutive readings (seconds):
count    98428.000000
mean         0.036620
std          0.034531
min          0.000000
25%          0.014932
50%          0.023909
75%          0.041888
max          0.484703
Name: Timestamp, dtype: float64

Median gap: 0.0239 seconds
That means roughly 41.8 readings per second

To span 10 real seconds, we need approximately 418 rows per window


In [9]:
# Rebuild windows using the CORRECT size: ~418 rows = ~10 real seconds
# Same logic as before, just the right window_size this time

window_size_correct = 418

print(f"Building {window_size_correct}-row (~10 second) sequences...")
print("This will produce far fewer sequences than before — expected, since")
print("each one now spans 10 real seconds instead of 0.24 seconds\n")

train_sequences_v2 = build_windows(train_df, train_participants, window_size=window_size_correct)
print(f"✅ Built {train_sequences_v2['sequence_id'].nunique():,} training sequences")
print(f"Total rows: {len(train_sequences_v2):,}")

test_sequences_v2 = build_windows(train_df, test_participants, window_size=window_size_correct)
print(f"✅ Built {test_sequences_v2['sequence_id'].nunique():,} test sequences")
print(f"Total rows: {len(test_sequences_v2):,}")

# Re-run the SAME variation check as before, on the corrected windows
variation_check_v2 = train_sequences_v2.groupby('sequence_id').agg({
    'Ambient_Temperature': lambda x: x.max() - x.min(),
    'Wrist_Skin_Temperature': lambda x: x.max() - x.min(),
    'GSR': lambda x: x.max() - x.min(),
    'Label': lambda x: x.nunique()
}).rename(columns={
    'Ambient_Temperature': 'AmbTemp_range',
    'Wrist_Skin_Temperature': 'SkinTemp_range',
    'GSR': 'GSR_range',
    'Label': 'unique_labels_in_window'
})

print("\nHow much do features vary WITHIN a corrected ~10-second window?")
print(variation_check_v2.describe())

print(f"\nWindows where Label changes within the window: "
      f"{(variation_check_v2['unique_labels_in_window'] > 1).sum():,} "
      f"out of {len(variation_check_v2):,} "
      f"({(variation_check_v2['unique_labels_in_window'] > 1).mean()*100:.2f}%)")

Building 418-row (~10 second) sequences...
This will produce far fewer sequences than before — expected, since
each one now spans 10 real seconds instead of 0.24 seconds

✅ Built 3,049 training sequences
Total rows: 1,274,482
✅ Built 692 test sequences
Total rows: 289,256

How much do features vary WITHIN a corrected ~10-second window?
       AmbTemp_range  SkinTemp_range    GSR_range  unique_labels_in_window
count    3049.000000     3049.000000  3049.000000              3049.000000
mean        0.158183        0.074385     0.113561                 1.074123
std         0.117328        0.078101     0.245965                 0.262013
min         0.000000        0.000000     0.001281                 1.000000
25%         0.100000        0.040000     0.007687                 1.000000
50%         0.100000        0.040000     0.023058                 1.000000
75%         0.200000        0.080000     0.093520                 1.000000
max         0.700000        1.160000     4.059144             

In [10]:
# Quick check: what does a label-CHANGING window actually look like?
# This tells us whether transitions are gradual or abrupt — useful
# context before training

changing_windows = variation_check_v2[variation_check_v2['unique_labels_in_window'] > 1]
example_seq_id = changing_windows.index[0]

example = train_sequences_v2[train_sequences_v2['sequence_id'] == example_seq_id]
print(f"Example transitioning sequence: {example_seq_id}")
print(example[['time_step', 'Ambient_Temperature', 'Wrist_Skin_Temperature', 'Label']]
      .iloc[::40])  # every 40th row so we can see the shape without 418 rows of output

Example transitioning sequence: participant_10_seq116
       time_step  Ambient_Temperature  Wrist_Skin_Temperature  Label
48488          0                 33.5                   34.37      2
48528         40                 33.5                   34.37      2
48568         80                 33.5                   34.39      2
48608        120                 33.4                   34.41      0
48648        160                 33.3                   34.39      0
48688        200                 33.3                   34.39      0
48728        240                 33.4                   34.41      0
48768        280                 33.3                   34.43      0
48808        320                 33.3                   34.41      0
48848        360                 33.3                   34.41      0
48888        400                 33.3                   34.45      0


In [11]:
# For every transitioning window, how much did the KEY FEATURES actually
# move, versus how many label categories did it jump across?

transition_detail = []
for seq_id in changing_windows.index:
    seq = train_sequences_v2[train_sequences_v2['sequence_id'] == seq_id]
    amb_range = seq['Ambient_Temperature'].max() - seq['Ambient_Temperature'].min()
    skin_range = seq['Wrist_Skin_Temperature'].max() - seq['Wrist_Skin_Temperature'].min()
    label_jump = seq['Label'].max() - seq['Label'].min()
    transition_detail.append({
        'sequence_id': seq_id,
        'amb_range': amb_range,
        'skin_range': skin_range,
        'label_jump': label_jump
    })

transition_df = pd.DataFrame(transition_detail)
print("Distribution of label jump size across all 226 transitioning windows:")
print(transition_df['label_jump'].value_counts().sort_index())

print("\nFeature movement for windows with label_jump == 1 (adjacent, plausible):")
print(transition_df[transition_df['label_jump'] == 1][['amb_range', 'skin_range']].describe())

print("\nFeature movement for windows with label_jump >= 2 (skips a category, suspicious):")
print(transition_df[transition_df['label_jump'] >= 2][['amb_range', 'skin_range']].describe())

Distribution of label jump size across all 226 transitioning windows:
label_jump
1    214
2      8
3      4
Name: count, dtype: int64

Feature movement for windows with label_jump == 1 (adjacent, plausible):
        amb_range  skin_range
count  214.000000  214.000000
mean     0.178037    0.062150
std      0.131206    0.065442
min      0.000000    0.020000
25%      0.100000    0.040000
50%      0.100000    0.040000
75%      0.200000    0.060000
max      0.700000    0.580000

Feature movement for windows with label_jump >= 2 (skips a category, suspicious):
       amb_range  skin_range
count  12.000000   12.000000
mean    0.141667    0.050000
std     0.099620    0.031334
min     0.000000    0.020000
25%     0.100000    0.035000
50%     0.100000    0.040000
75%     0.200000    0.055000
max     0.400000    0.100000


In [12]:
# DeepEcho needs sequences indexed properly and entity columns clear

ts_columns_for_model = [
    'Wrist_Skin_Temperature', 'GSR', 'Ambient_Temperature',
    'Ambient_Humidity', 'Radiation-Temp', 'PCE-Ambient-Temp',
    'Heart_Rate', 'Label'
]

context_columns = ['Age', 'Gender', 'Weight', 'Clothing-Level']

print(f"Sequence variables (evolve over time): {ts_columns_for_model}")
print(f"Context variables (fixed per sequence): {context_columns}")
print(f"\nTotal training sequences: {train_sequences_v2['sequence_id'].nunique():,}")

# DeepEcho PARModel setup
par_data = train_sequences_v2[
    ['sequence_id', 'time_step'] + ts_columns_for_model + context_columns
].copy()

print("\nSample of data going into PARModel:")
print(par_data.head(5))

Sequence variables (evolve over time): ['Wrist_Skin_Temperature', 'GSR', 'Ambient_Temperature', 'Ambient_Humidity', 'Radiation-Temp', 'PCE-Ambient-Temp', 'Heart_Rate', 'Label']
Context variables (fixed per sequence): ['Age', 'Gender', 'Weight', 'Clothing-Level']

Total training sequences: 3,049

Sample of data going into PARModel:
           sequence_id  time_step  Wrist_Skin_Temperature       GSR  \
0  participant_10_seq0          0                   30.05  0.198558   
1  participant_10_seq0          1                   30.05  0.198558   
2  participant_10_seq0          2                   30.05  0.198558   
3  participant_10_seq0          3                   30.05  0.198558   
4  participant_10_seq0          4                   30.07  0.198558   

   Ambient_Temperature  Ambient_Humidity  Radiation-Temp  PCE-Ambient-Temp  \
0                 20.4              34.0            19.6              19.7   
1                 20.4              34.0            19.6              19.7   
2     

In [13]:
print("Training DeepEcho PARModel...")
print(f"3,049 sequences x 418 time steps x 8 sequence variables + 4 context variables")
print("This is a substantial training task — likely 20-40 minutes on CPU\n")

par_model = PARModel(
    epochs=128,
    cuda=True,
    verbose=True
)

par_model.fit(
    data=par_data,
    entity_columns=['sequence_id'],
    context_columns=context_columns,
    sequence_index='time_step'
)

print("\n✅ PARModel fitted!")

Training DeepEcho PARModel...
3,049 sequences x 418 time steps x 8 sequence variables + 4 context variables
This is a substantial training task — likely 20-40 minutes on CPU



Loss (-00.01): 100%|██████████| 128/128 [1:08:11<00:00, 31.96s/it]



✅ PARModel fitted!


In [14]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Current device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")

CUDA available: True
Current device: NVIDIA A100-SXM4-40GB


In [15]:
print("Generating synthetic sequences from PARModel...")

synthetic_sequences = par_model.sample(
    num_entities=len(train_sequences_v2['sequence_id'].unique())
)

print(f"✅ Generated {synthetic_sequences['sequence_id'].nunique():,} synthetic sequences")
print(f"Total rows: {len(synthetic_sequences):,}")
print("\nFirst sequence preview:")
print(synthetic_sequences.head(10))

Generating synthetic sequences from PARModel...


  3%|▎         | 84/3049 [06:25<3:46:38,  4.59s/it]


KeyboardInterrupt: 

In [16]:
# Stop the current generation, restart with a smaller, faster sample
# to validate the approach before committing 4 hours

print("Generating a SMALLER synthetic sample first — 300 sequences —")
print("to validate quality before committing to the full generation run")
print("Expected time: ~23 minutes\n")

synthetic_sequences_small = par_model.sample(num_entities=300)

print(f"✅ Generated {synthetic_sequences_small['sequence_id'].nunique():,} synthetic sequences")
print(f"Total rows: {len(synthetic_sequences_small):,}")
print("\nFirst sequence preview:")
print(synthetic_sequences_small.head(10))

Generating a SMALLER synthetic sample first — 300 sequences —
to validate quality before committing to the full generation run
Expected time: ~23 minutes



100%|██████████| 300/300 [22:46<00:00,  4.56s/it]

✅ Generated 300 synthetic sequences
Total rows: 125,400

First sequence preview:
   sequence_id  Wrist_Skin_Temperature       GSR  Ambient_Temperature  \
0            0               33.826413  0.579320            26.886290   
1            0               32.451218  0.433926            28.424962   
2            0               33.029117  0.433064            27.676737   
3            0               32.278852  0.595844            26.477454   
4            0               32.490559  0.430343            26.886290   
5            0               32.063333  0.027049            27.751580   
6            0               31.294909  0.053068            26.609207   
7            0               32.436958  0.044614            26.386266   
8            0               31.116847  0.184205            26.498372   
9            0               31.763662 -0.061571            27.321021   

   Ambient_Humidity  Radiation-Temp  PCE-Ambient-Temp  Heart_Rate     Label  \
0         31.459692       25.879537 

In [17]:
# Round and clip Label to the valid discrete range
synthetic_sequences_small['Label'] = (
    synthetic_sequences_small['Label'].round().clip(-3, 3).astype(int)
)

print("Label distribution — synthetic (300 sequences, all time steps):")
print(synthetic_sequences_small['Label'].value_counts().sort_index())

# Compare REAL vs SYNTHETIC feature statistics — same fidelity check
# we ran on ForestDiffusion, so the comparison is apples-to-apples
key_features_ts = ['Ambient_Temperature', 'Wrist_Skin_Temperature',
                     'Radiation-Temp', 'PCE-Ambient-Temp',
                     'Ambient_Humidity', 'GSR', 'Heart_Rate']

# Pull the matching real sequences for comparison
real_sample_ids = train_sequences_v2['sequence_id'].unique()[:300]
real_compare = train_sequences_v2[train_sequences_v2['sequence_id'].isin(real_sample_ids)]

comparison_par = pd.DataFrame({
    'Real_mean': real_compare[key_features_ts].mean(),
    'Synthetic_mean': synthetic_sequences_small[key_features_ts].mean(),
    'Real_std': real_compare[key_features_ts].std(),
    'Synthetic_std': synthetic_sequences_small[key_features_ts].std(),
}).round(3)

comparison_par['std_ratio'] = (
    comparison_par['Synthetic_std'] / comparison_par['Real_std']
).round(2)

print("\nFidelity check — PARModel vs Real:")
print(comparison_par)

Label distribution — synthetic (300 sequences, all time steps):
Label
-3       94
-2     2195
-1    16483
 0    50329
 1    42884
 2    12341
 3     1074
Name: count, dtype: int64

Fidelity check — PARModel vs Real:
                        Real_mean  Synthetic_mean  Real_std  Synthetic_std  \
Ambient_Temperature        26.023          27.264     4.016          1.985   
Wrist_Skin_Temperature     32.603          33.424     1.498          1.310   
Radiation-Temp             24.294          25.842     3.239          1.600   
PCE-Ambient-Temp           24.059          25.715     3.041          1.537   
Ambient_Humidity           26.062          31.196     7.194          4.669   
GSR                         0.257           0.520     0.123          0.452   
Heart_Rate                 72.751          65.124    22.280          6.228   

                        std_ratio  
Ambient_Temperature          0.49  
Wrist_Skin_Temperature       0.87  
Radiation-Temp               0.49  
PCE-Ambient-Tem

In [20]:
help(par_model.sample)

Help on method sample in module deepecho.models.base:

sample(num_entities=None, context=None, sequence_length=None) method of deepecho.models.par.PARModel instance
    Sample a dataframe containing time series data.

    Args:
        num_entities (int):
            The number of entities to sample.
        context (pd.DataFrame):
            Context values to use when sampling.
        sequence_length (int or None):
            If given, force sequences to be of the indicated length.
            If ``None`` (default), sample sequences of the same length
            as the original dataset.

    Returns:
        pd.DataFrame:
            A DataFrame which resembles the original dataframe where (1) the
            entity column(s) are arbitrarily generated, (2) the context
            column(s) are resampled from the original data, and (3) the data
            columns containing the time series comes from the conditional
            time series model.



In [21]:
# Did the model happen to generate sequences mostly for "warm-leaning"
# context profiles, which would explain the label skew without it
# being a model failure at all?

print("Context variable summary — REAL data (all training sequences):")
context_summary_real = train_sequences_v2.groupby('sequence_id')[
    ['Age', 'Gender', 'Weight', 'Clothing-Level']
].first()
print(context_summary_real.describe())

print("\nContext variable summary — SYNTHETIC (300 generated sequences):")
context_summary_synth = synthetic_sequences_small.groupby('sequence_id')[
    ['Age', 'Gender', 'Weight', 'Clothing-Level']
].first()
print(context_summary_synth.describe())

Context variable summary — REAL data (all training sequences):
               Age       Gender       Weight  Clothing-Level
count  3049.000000  3049.000000  3049.000000     3049.000000
mean     24.597901     0.613644    73.873532        0.591620
std       2.618142     0.486994    16.593536        0.057348
min      20.000000     0.000000    54.200000        0.450000
25%      23.000000     0.000000    59.700000        0.570000
50%      25.000000     1.000000    67.000000        0.570000
75%      25.000000     1.000000    88.100000        0.610000
max      30.000000     1.000000   104.900000        0.690000

Context variable summary — SYNTHETIC (300 generated sequences):
              Age      Gender      Weight  Clothing-Level
count  300.000000  300.000000  300.000000      300.000000
mean    24.520000    0.630000   73.385333        0.585600
std      2.643146    0.483611   15.934046        0.057477
min     20.000000    0.000000   54.200000        0.450000
25%     23.000000    0.000000   5

In [23]:
# Does Clothing-Level (a plausible proxy for cold-sensitivity) actually
# relate to who reports Cold in the REAL data? If yes, conditioning
# generation on it could plausibly help.

cold_reporters = train_sequences_v2[train_sequences_v2['Label'] <= -2]['sequence_id'].unique()
train_sequences_v2['is_cold_sequence'] = train_sequences_v2['sequence_id'].isin(cold_reporters)

context_vs_cold = train_sequences_v2.groupby('sequence_id').agg({
    'Clothing-Level': 'first',
    'Age': 'first',
    'is_cold_sequence': 'first'
})

print("Clothing-Level and Age by whether sequence contains a Cold reading:")
print(context_vs_cold.groupby('is_cold_sequence')[['Clothing-Level', 'Age']].describe())

Clothing-Level and Age by whether sequence contains a Cold reading:
                 Clothing-Level                                              \
                          count      mean       std   min   25%   50%   75%   
is_cold_sequence                                                              
False                    2644.0  0.592587  0.056675  0.45  0.57  0.57  0.61   
True                      405.0  0.585309  0.061263  0.45  0.57  0.57  0.61   

                           Age                                               \
                   max   count       mean       std   min   25%   50%   75%   
is_cold_sequence                                                              
False             0.69  2644.0  24.686838  2.606977  20.0  23.0  25.0  26.0   
True              0.69   405.0  24.017284  2.619840  20.0  22.0  25.0  25.0   

                        
                   max  
is_cold_sequence        
False             30.0  
True              29.0  


In [24]:
print("="*70)
print("TIME-SERIES INVESTIGATION — CONCLUSION")
print("="*70)
print("""
1. WINDOWING ERROR IDENTIFIED AND CORRECTED:
   Initial 10-row windows spanned only ~0.24 real seconds (AutoTherm
   samples at ~42Hz), showing zero feature variation and 0.16% label
   transitions. Corrected to 418-row windows (~10 real seconds),
   yielding 7.41% label transitions, 94.7% of which were plausible
   single-category adjacent shifts.

2. DEEPECHO PARMODEL TRAINED SUCCESSFULLY:
   3,049 training sequences, 128 epochs, converged (final loss ~-0.01).

3. GENERATION REVEALED REGRESSION-TO-THE-MEAN:
   Feature std ratios 0.28-0.87 (too narrow) across 6/7 key features.
   Label distribution collapsed toward dominant classes:
   Cold (-3) reduced to 0.07% synthetic vs 4.50% real.
   Hot (+3) reduced to 0.86% synthetic vs 13.75% real.

4. RULED OUT TWO ALTERNATIVE EXPLANATIONS:
   - Context sampling bias: ruled out (synthetic context closely
     matches real population distribution).
   - Context-conditioning fix: ruled out (Clothing-Level/Age show
     negligible association with Cold-class membership in real data,
     and DeepEcho resamples context independently per documentation).

5. CONCLUSION:
   Regression-to-the-mean is intrinsic to PARModel's autoregressive
   objective on this dataset, not a correctable sampling or
   conditioning artefact. Downstream TSTR/augmented evaluation not
   pursued given this fidelity failure — consistent with the
   methodological standard applied to ForestDiffusion.
""")

TIME-SERIES INVESTIGATION — CONCLUSION

1. WINDOWING ERROR IDENTIFIED AND CORRECTED:
   Initial 10-row windows spanned only ~0.24 real seconds (AutoTherm
   samples at ~42Hz), showing zero feature variation and 0.16% label
   transitions. Corrected to 418-row windows (~10 real seconds),
   yielding 7.41% label transitions, 94.7% of which were plausible
   single-category adjacent shifts.

2. DEEPECHO PARMODEL TRAINED SUCCESSFULLY:
   3,049 training sequences, 128 epochs, converged (final loss ~-0.01).

3. GENERATION REVEALED REGRESSION-TO-THE-MEAN:
   Feature std ratios 0.28-0.87 (too narrow) across 6/7 key features.
   Label distribution collapsed toward dominant classes:
   Cold (-3) reduced to 0.07% synthetic vs 4.50% real.
   Hot (+3) reduced to 0.86% synthetic vs 13.75% real.

4. RULED OUT TWO ALTERNATIVE EXPLANATIONS:
   - Context sampling bias: ruled out (synthetic context closely
     matches real population distribution).
   - Context-conditioning fix: ruled out (Clothing-Leve

In [25]:
ts_investigation_summary = pd.DataFrame({
    'Stage': [
        'Initial windowing (10 rows)',
        'Corrected windowing (418 rows)',
        'PARModel training',
        'PARModel generation - feature fidelity',
        'PARModel generation - label distribution',
    ],
    'Finding': [
        'Zero feature variation, 0.16% label transitions',
        '7.41% label transitions, 94.7% single-step adjacent',
        'Converged successfully, 128 epochs, loss ~-0.01',
        'Std ratios 0.28-0.87 (too narrow), GSR 3.67 (too wide)',
        'Cold 0.07% synthetic vs 4.50% real; Hot 0.86% vs 13.75%',
    ]
})

print(ts_investigation_summary.to_string(index=False))
ts_investigation_summary.to_csv('wk6_parmodel_investigation_summary.csv', index=False)
print("\n✅ Summary saved")

                                   Stage                                                 Finding
             Initial windowing (10 rows)         Zero feature variation, 0.16% label transitions
          Corrected windowing (418 rows)     7.41% label transitions, 94.7% single-step adjacent
                       PARModel training         Converged successfully, 128 epochs, loss ~-0.01
  PARModel generation - feature fidelity  Std ratios 0.28-0.87 (too narrow), GSR 3.67 (too wide)
PARModel generation - label distribution Cold 0.07% synthetic vs 4.50% real; Hot 0.86% vs 13.75%

✅ Summary saved
